# 🧠 Speech Emotion Recognition — Training

## 🎯 Modèles entraînés
1. **CNN + BiLSTM** : Architecture hybride avec features acoustiques manuelles
2. **Wav2Vec2-base** : Fine-tuning du modèle pré-entraîné Facebook (Self-Supervised Learning)
3. **HuBERT-base** : Fine-tuning du modèle pré-entraîné Facebook (Hidden-Unit BERT)

## ⚙️ Optimisations RTX 3050 (6 GB VRAM)
- **Mixed Precision (FP16)** pour réduire la VRAM
- **Gradient Accumulation** pour simuler des batch sizes plus grands
- **Feature Extractor Frozen** : Seule la tête de classification est entraînée initialement
- **Batch sizes adaptés** : 8-16 pour transformer, 32 pour CNN

## 📚 Références
- Baevski et al. (2020). *wav2vec 2.0: A Framework for Self-Supervised Learning of Speech Representations*
- Hsu et al. (2021). *HuBERT: Self-Supervised Speech Representation Learning by Masked Prediction of Hidden Units*
- Zhao et al. (2019). *Speech emotion recognition using deep 1D & 2D CNN LSTM networks*

In [ ]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

# Supprimer l'avertissement HuggingFace Hub (accès anonyme suffisant pour modèles publics)
os.environ.setdefault('HF_HUB_VERBOSITY', 'error')
import logging
logging.getLogger('huggingface_hub').setLevel(logging.ERROR)

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from tqdm import tqdm
import time

# Vérification GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    # Optimisations mémoire
    torch.backends.cudnn.benchmark = True
    torch.cuda.empty_cache()


In [ ]:
# Créer les dossiers de sortie
os.makedirs('models', exist_ok=True)
os.makedirs('histories', exist_ok=True)

---
## 🎤 Modèle 1 : CNN + BiLSTM

Architecture hybride :
- **Conv1D** : Extraction de patterns locaux dans les features
- **BiLSTM** : Capture des dépendances temporelles bidirectionnelles
- **Features** : MFCC + Chroma + ZCR + RMS + Spectral (57 features)

In [ ]:
# Charger les données préprocessées
cnn_data = np.load('preprocessed/cnn_bilstm_data.npz')

X_train_cnn = torch.FloatTensor(cnn_data['X_train'])
X_val_cnn = torch.FloatTensor(cnn_data['X_val'])
X_test_cnn = torch.FloatTensor(cnn_data['X_test'])
y_train_cnn = torch.LongTensor(cnn_data['y_train'])
y_val_cnn = torch.LongTensor(cnn_data['y_val'])
y_test_cnn = torch.LongTensor(cnn_data['y_test'])

num_classes = len(np.unique(cnn_data['y_train']))
input_timesteps = X_train_cnn.shape[1]
input_features = X_train_cnn.shape[2]

print(f'Input shape: ({input_timesteps}, {input_features})')
print(f'Num classes: {num_classes}')
print(f'Train: {len(X_train_cnn)} | Val: {len(X_val_cnn)} | Test: {len(X_test_cnn)}')

In [ ]:
class EmotionDataset(Dataset):
    """Dataset PyTorch pour les features acoustiques."""
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# DataLoaders
BATCH_SIZE_CNN = 32

train_loader_cnn = DataLoader(EmotionDataset(X_train_cnn, y_train_cnn), 
                              batch_size=BATCH_SIZE_CNN, shuffle=True, pin_memory=True)
val_loader_cnn = DataLoader(EmotionDataset(X_val_cnn, y_val_cnn), 
                            batch_size=BATCH_SIZE_CNN, shuffle=False, pin_memory=True)
test_loader_cnn = DataLoader(EmotionDataset(X_test_cnn, y_test_cnn), 
                             batch_size=BATCH_SIZE_CNN, shuffle=False, pin_memory=True)
print(f'Batch size: {BATCH_SIZE_CNN}')
print(f'Train batches: {len(train_loader_cnn)}')

In [ ]:
class CNN_BiLSTM(nn.Module):
    """Architecture CNN + BiLSTM pour la reconnaissance d'\u00e9motions."""
    
    def __init__(self, input_features, num_classes):
        super().__init__()
        
        # Bloc CNN 1
        self.conv1 = nn.Sequential(
            nn.Conv1d(input_features, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.3)
        )
        
        # Bloc CNN 2
        self.conv2 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.3)
        )
        
        # Bloc CNN 3
        self.conv3 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.3)
        )
        
        # BiLSTM
        self.bilstm = nn.LSTM(
            input_size=256, hidden_size=128,
            num_layers=2, batch_first=True,
            bidirectional=True, dropout=0.3
        )
        
        # Attention mechanism
        self.attention = nn.Sequential(
            nn.Linear(256, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        # x: (batch, timesteps, features)
        x = x.permute(0, 2, 1)  # -> (batch, features, timesteps) pour Conv1d
        
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        
        x = x.permute(0, 2, 1)  # -> (batch, seq_len, channels) pour LSTM
        
        lstm_out, _ = self.bilstm(x)  # (batch, seq_len, 256)
        
        # Attention
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)  # (batch, seq_len, 1)
        context = torch.sum(attn_weights * lstm_out, dim=1)  # (batch, 256)
        
        out = self.classifier(context)
        return out

model_cnn = CNN_BiLSTM(input_features, num_classes).to(device)
total_params = sum(p.numel() for p in model_cnn.parameters())
trainable_params = sum(p.numel() for p in model_cnn.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(model_cnn)

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs, lr=1e-3, patience=10, model_name='model'):
    """Boucle d'entraînement générique avec mixed precision."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    scaler = GradScaler('cuda')
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(num_epochs):
        start = time.time()
        
        # --- TRAIN ---
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            with autocast('cuda'):
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item() * X_batch.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += (preds == y_batch).sum().item()
            train_total += y_batch.size(0)
        
        train_loss /= train_total
        train_acc = train_correct / train_total
        
        # --- VALIDATION ---
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                with autocast('cuda'):
                    outputs = model(X_batch)
                    loss = criterion(outputs, y_batch)
                
                val_loss += loss.item() * X_batch.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == y_batch).sum().item()
                val_total += y_batch.size(0)
        
        val_loss /= val_total
        val_acc = val_correct / val_total
        
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        
        elapsed = time.time() - start
        print(f'Epoch {epoch+1:3d}/{num_epochs} | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | '
              f'LR: {current_lr:.6f} | {elapsed:.1f}s')
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), f'models/{model_name}_best.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'\nEarly stopping at epoch {epoch+1}')
                break
    
    # Charger le meilleur modèle
    model.load_state_dict(torch.load(f'models/{model_name}_best.pth', weights_only=True))
    
    # Sauvegarder l'historique
    with open(f'histories/{model_name}_history.json', 'w') as f:
        json.dump(history, f)
    
    return history

print('Fonction d\'entraînement définie.')

In [ ]:
# Entraînement CNN+BiLSTM
print('=' * 70)
print('ENTRA\u00ceNEMENT : CNN + BiLSTM')
print('=' * 70)

history_cnn = train_model(
    model=model_cnn,
    train_loader=train_loader_cnn,
    val_loader=val_loader_cnn,
    num_epochs=60,
    lr=1e-3,
    patience=10,
    model_name='cnn_bilstm'
)

torch.cuda.empty_cache()
print('\nMod\u00e8le CNN+BiLSTM entra\u00een\u00e9 et sauvegard\u00e9.')

---
## 🌐 Modèle 2 : Wav2Vec2

**Wav2Vec 2.0** (Baevski et al., 2020) :
- Pré-entraîné sur 960h de LibriSpeech (self-supervised)
- Encode le contexte temporel via un Transformer
- Fine-tuning de la tête de classification uniquement (pour RTX 3050)
- Modèle **base** (~95M params, feature extractor frozen = ~2M trainable)

In [ ]:
from transformers import Wav2Vec2Model, Wav2Vec2Config

# Charger données transformer
trans_data = np.load('preprocessed/transformer_data.npz')

X_train_wav = trans_data['X_train']
X_val_wav = trans_data['X_val']
X_test_wav = trans_data['X_test']
y_train_wav = trans_data['y_train']
y_val_wav = trans_data['y_val']
y_test_wav = trans_data['y_test']

print(f'Train: {X_train_wav.shape} | Val: {X_val_wav.shape} | Test: {X_test_wav.shape}')

In [ ]:
class WaveformDataset(Dataset):
    """Dataset PyTorch pour les waveforms brutes."""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Batch sizes réduits pour RTX 3050
BATCH_SIZE_TRANSFORMER = 8
GRAD_ACCUM_STEPS = 4  # Effective batch size = 8 * 4 = 32

train_loader_wav = DataLoader(WaveformDataset(X_train_wav, y_train_wav),
                              batch_size=BATCH_SIZE_TRANSFORMER, shuffle=True, pin_memory=True)
val_loader_wav = DataLoader(WaveformDataset(X_val_wav, y_val_wav),
                            batch_size=BATCH_SIZE_TRANSFORMER, shuffle=False, pin_memory=True)
test_loader_wav = DataLoader(WaveformDataset(X_test_wav, y_test_wav),
                             batch_size=BATCH_SIZE_TRANSFORMER, shuffle=False, pin_memory=True)

print(f'Batch size: {BATCH_SIZE_TRANSFORMER} (effective: {BATCH_SIZE_TRANSFORMER * GRAD_ACCUM_STEPS})')

In [ ]:
class Wav2Vec2ForSER(nn.Module):
    """Wav2Vec2 avec t\u00eate de classification pour SER."""
    
    def __init__(self, num_classes, freeze_feature_extractor=True):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base')
        
        # Geler le feature extractor CNN (\u00e9conomie de VRAM)
        if freeze_feature_extractor:
            self.wav2vec2.feature_extractor._freeze_parameters()
            # Geler aussi les premi\u00e8res couches du transformer encoder
            for i, layer in enumerate(self.wav2vec2.encoder.layers):
                if i < 8:  # Geler les 8 premi\u00e8res couches (sur 12)
                    for param in layer.parameters():
                        param.requires_grad = False
        
        # T\u00eate de classification
        hidden_size = self.wav2vec2.config.hidden_size  # 768
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, input_values):
        outputs = self.wav2vec2(input_values)
        hidden_states = outputs.last_hidden_state  # (batch, seq_len, 768)
        pooled = hidden_states.mean(dim=1)  # Mean pooling
        logits = self.classifier(pooled)
        return logits

model_w2v = Wav2Vec2ForSER(num_classes, freeze_feature_extractor=True).to(device)

total_params = sum(p.numel() for p in model_w2v.parameters())
trainable_params = sum(p.numel() for p in model_w2v.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(f'Frozen params: {total_params - trainable_params:,}')

In [ ]:
def train_transformer(model, train_loader, val_loader, num_epochs, lr=1e-4, 
                      patience=8, model_name='transformer', grad_accum_steps=4):
    """Boucle d'entraînement pour modèles transformer avec gradient accumulation."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), 
                            lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
    scaler = GradScaler('cuda')
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
    best_val_loss = float('inf')
    patience_counter = 0
    
    for epoch in range(num_epochs):
        start = time.time()
        
        # --- TRAIN ---
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        optimizer.zero_grad()
        
        for step, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            with autocast('cuda'):
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch) / grad_accum_steps
            
            scaler.scale(loss).backward()
            
            if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            train_loss += loss.item() * grad_accum_steps * X_batch.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += (preds == y_batch).sum().item()
            train_total += y_batch.size(0)
        
        train_loss /= train_total
        train_acc = train_correct / train_total
        scheduler.step()
        
        # --- VALIDATION ---
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                with autocast('cuda'):
                    outputs = model(X_batch)
                    loss = criterion(outputs, y_batch)
                
                val_loss += loss.item() * X_batch.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == y_batch).sum().item()
                val_total += y_batch.size(0)
        
        val_loss /= val_total
        val_acc = val_correct / val_total
        current_lr = optimizer.param_groups[0]['lr']
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        
        elapsed = time.time() - start
        print(f'Epoch {epoch+1:3d}/{num_epochs} | '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | '
              f'LR: {current_lr:.6f} | {elapsed:.1f}s')
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), f'models/{model_name}_best.pth')
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'\nEarly stopping at epoch {epoch+1}')
                break
    
    model.load_state_dict(torch.load(f'models/{model_name}_best.pth', weights_only=True))
    
    with open(f'histories/{model_name}_history.json', 'w') as f:
        json.dump(history, f)
    
    return history

print('Fonction d\'entraînement transformer définie.')

In [ ]:
# Entraînement Wav2Vec2
print('=' * 70)
print('ENTRA\u00ceNEMENT : Wav2Vec2')
print('=' * 70)

history_w2v = train_transformer(
    model=model_w2v,
    train_loader=train_loader_wav,
    val_loader=val_loader_wav,
    num_epochs=30,
    lr=2e-5,
    patience=8,
    model_name='wav2vec2',
    grad_accum_steps=GRAD_ACCUM_STEPS
)

torch.cuda.empty_cache()
print('\nMod\u00e8le Wav2Vec2 entra\u00een\u00e9 et sauvegard\u00e9.')

---
## 🎭 Modèle 3 : HuBERT

**HuBERT** (Hsu et al., 2021) :
- Pré-entraîné sur LibriSpeech 960h
- Apprentissage auto-supervisé par prédiction d’unités cachées (k-means sur MFCCs)
- Architecture Transformer similaire à Wav2Vec2
- Souvent meilleur sur les tâches de classification audio

In [ ]:
from transformers import HubertModel

# Libérer la VRAM du modèle précédent
del model_w2v
torch.cuda.empty_cache()

class HuBERTForSER(nn.Module):
    """HuBERT avec t\u00eate de classification pour SER."""
    
    def __init__(self, num_classes, freeze_feature_extractor=True):
        super().__init__()
        self.hubert = HubertModel.from_pretrained('facebook/hubert-base-ls960')
        
        # Geler le feature extractor et les premi\u00e8res couches
        if freeze_feature_extractor:
            self.hubert.feature_extractor._freeze_parameters()
            for i, layer in enumerate(self.hubert.encoder.layers):
                if i < 8:
                    for param in layer.parameters():
                        param.requires_grad = False
        
        hidden_size = self.hubert.config.hidden_size  # 768
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, input_values):
        outputs = self.hubert(input_values)
        hidden_states = outputs.last_hidden_state  # (batch, seq_len, 768)
        pooled = hidden_states.mean(dim=1)  # Mean pooling
        logits = self.classifier(pooled)
        return logits

model_hubert = HuBERTForSER(num_classes, freeze_feature_extractor=True).to(device)

total_params = sum(p.numel() for p in model_hubert.parameters())
trainable_params = sum(p.numel() for p in model_hubert.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(f'Frozen params: {total_params - trainable_params:,}')

In [ ]:
# Entraînement HuBERT
print('=' * 70)
print('ENTRA\u00ceNEMENT : HuBERT')
print('=' * 70)

history_hubert = train_transformer(
    model=model_hubert,
    train_loader=train_loader_wav,
    val_loader=val_loader_wav,
    num_epochs=30,
    lr=2e-5,
    patience=8,
    model_name='hubert',
    grad_accum_steps=GRAD_ACCUM_STEPS
)

torch.cuda.empty_cache()
print('\nMod\u00e8le HuBERT entra\u00een\u00e9 et sauvegard\u00e9.')

---
## 📊 Évaluation Rapide sur le Test Set

In [ ]:
def evaluate_model(model, test_loader):
    """Évaluer un modèle sur le test set."""
    model.eval()
    all_preds, all_labels = [], []
    test_loss, test_correct, test_total = 0, 0, 0
    criterion = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            with autocast('cuda'):
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
            
            test_loss += loss.item() * X_batch.size(0)
            _, preds = torch.max(outputs, 1)
            test_correct += (preds == y_batch).sum().item()
            test_total += y_batch.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    test_loss /= test_total
    test_acc = test_correct / test_total
    
    return test_loss, test_acc, np.array(all_preds), np.array(all_labels)

# Sauvegarder les prédictions pour le notebook Results
os.makedirs('predictions', exist_ok=True)

# --- CNN+BiLSTM ---
model_cnn.load_state_dict(torch.load('models/cnn_bilstm_best.pth', weights_only=True))
loss_cnn, acc_cnn, preds_cnn, labels_cnn = evaluate_model(model_cnn, test_loader_cnn)
np.savez('predictions/cnn_bilstm_preds.npz', preds=preds_cnn, labels=labels_cnn)
print(f'CNN+BiLSTM  | Loss: {loss_cnn:.4f} | Accuracy: {acc_cnn:.4f}')

# --- Wav2Vec2 ---
model_w2v_eval = Wav2Vec2ForSER(num_classes).to(device)
model_w2v_eval.load_state_dict(torch.load('models/wav2vec2_best.pth', weights_only=True))
loss_w2v, acc_w2v, preds_w2v, labels_w2v = evaluate_model(model_w2v_eval, test_loader_wav)
np.savez('predictions/wav2vec2_preds.npz', preds=preds_w2v, labels=labels_w2v)
print(f'Wav2Vec2    | Loss: {loss_w2v:.4f} | Accuracy: {acc_w2v:.4f}')
del model_w2v_eval
torch.cuda.empty_cache()

# --- HuBERT ---
model_hubert.load_state_dict(torch.load('models/hubert_best.pth', weights_only=True))
loss_hub, acc_hub, preds_hub, labels_hub = evaluate_model(model_hubert, test_loader_wav)
np.savez('predictions/hubert_preds.npz', preds=preds_hub, labels=labels_hub)
print(f'HuBERT      | Loss: {loss_hub:.4f} | Accuracy: {acc_hub:.4f}')

print('\nToutes les prédictions sauvegardées dans predictions/')

In [ ]:
# Résumé final
print('=' * 60)
print('R\u00c9SUM\u00c9 DE L\'ENTRA\u00ceNEMENT')
print('=' * 60)
print(f'\n{"Mod\u00e8le":<15} {"Test Loss":<12} {"Test Accuracy":<15}')
print('-' * 42)
print(f'{"CNN+BiLSTM":<15} {loss_cnn:<12.4f} {acc_cnn:<15.4f}')
print(f'{"Wav2Vec2":<15} {loss_w2v:<12.4f} {acc_w2v:<15.4f}')
print(f'{"HuBERT":<15} {loss_hub:<12.4f} {acc_hub:<15.4f}')
print(f'\nFichiers sauv\u00e9gard\u00e9s :')
print(f'  models/*.pth           - Poids des mod\u00e8les')
print(f'  histories/*.json       - Historiques d\'entra\u00eenement')
print(f'  predictions/*.npz      - Pr\u00e9dictions test')